# 3. Select pairs, extract environments, and inspect forces
**Input:** your dump, explicit units, optional data file.
**Output:** candidate assessments, a finite pair/shell, source forces and optional exports.
**Modules:** io.lammps → chemistry.cpd_n4t.pairs / chemistry.cpd_n4t.selection → pairs.structures → chemistry.cpd_n4t.visualization / structure.alignment.

Automatic selection, angle assessment and plane overlays are **CPD/dienophile-specific**. For other chemistry, provide two complete molecule-ID lists and a relative lattice shift. This supports geometric extraction, not general reaction classification. No saved reports are required.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
from io import StringIO
from dataclasses import asdict
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ase.io import read, write
from MartiniFuncs.io.lammps import read_lammps_frame
from MartiniFuncs.trajectory.analysis import resolve_run_files
from MartiniFuncs.chemistry.cpd_n4t import pairs as cpd_pairs
from MartiniFuncs.pairs import structures
from MartiniFuncs.statistics import md
from MartiniFuncs.chemistry.cpd_n4t.selection import PairCriteria, assess_prereactive_pairs
from MartiniFuncs.chemistry.cpd_n4t.visualization import plot_cluster, plot_pair_geometry
from MartiniFuncs.structure.alignment import align_geometry

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None
DATA_FILE = None
UNITS = "metal"
TIMESTEP_PS = 0.0005
TYPE_SYMBOLS = None
FRAME_INDEX = 0
BOND_SCALE = 1.3
REACTIVE_RADIUS = 3.25
SHELL_RADIUS = 3.0
CRITERIA = PairCriteria()  # optional maxima for angles, forming distance, imbalance
SELECTED_CANDIDATE = 0
MANUAL_PAIR_IDS = None  # ([persistent IDs of molecule A], [IDs of molecule B])
MANUAL_SHIFT = (0, 0, 0)  # integer lattice translation of A relative to B
REFERENCE_EXTXYZ = None  # optional same-identity finite pair
OUTPUT_DIR = RUN_DIR / "analysis"
WRITE_FILES = False
OVERWRITE = False


### Mathematics: frames, time, units and molecular identity
Let $f$ be the zero-based source frame, $s_f$ its LAMMPS timestep, and $h=$ `TIMESTEP_PS` (ps per MD step). The displayed time is
$$t_f=s_fh.$$
`FRAME_INDEX` / `FRAME_INDICES` select $f$, not $s_f$. Where present, `FRAME_STRIDE` $q$ retains $f\bmod q=0$; `MAX_FRAMES` $F$ reads only $0\le f<F$. `STARTFRAME`, `ENDFRAME` or each `WINDOWS` entry $(a,b)$ keeps retained frames with $a\le f<b$; `None` means the available end. The lag spacing is $\Delta t=t_{j+1}-t_j$ between **retained** samples, not $h$. Paths and write/display switches do not enter these equations.

`UNITS` controls conversion on reading. In these workflows coordinates are Å, masses are atomic mass units, source forces are eV/Å, and reported velocities are Å/ps. `TYPE_SYMBOLS` sets the type-to-element mapping used to infer atomic properties when needed; an incorrect mapping can change masses and bond radii.

Where `BOND_SCALE` $s$ is used (default 1.3), atoms are connected when their periodic-image separation satisfies
$$\|\mathbf r_j+\mathbf nH-\mathbf r_i\|\le s(r_i^{\rm cov}+r_j^{\rm cov}).$$
$H$ contains cell vectors as rows, $\mathbf n$ is an integer image shift, and $r^{\rm cov}$ is the element's covalent radius (Å). Molecules are connected components of this inferred graph; increasing $s$ can merge close molecules. Persistent atom IDs are labels, not coordinate-array row indices.


In [ ]:
files = resolve_run_files(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE, require_input=False)
print("Selected files:", files)
frame = read_lammps_frame(files["dump"], index=FRAME_INDEX, units=UNITS,
                          input_datafile=files["data"], type_symbols=TYPE_SYMBOLS)
atoms = frame.atoms
print(f"Frame {frame.index}; step {frame.timestep}; time {frame.timestep*TIMESTEP_PS:g} ps")


## Assess candidates
The distance screen uses CPD COM to reactive-C=C midpoint. PairCriteria adds optional upper limits; every enabled criterion must pass. Axis angle compares reactive axes; plane angle compares fitted normals; approach angle compares site separation with the diene normal. All angles are unsigned 0–90 degrees. Approach is position, not motion.

`forming_bonds` contains proposed persistent-ID pairings. `forming_bonds_local` indexes the extracted pair; `forming_bonds_frame_local` indexes the full frame. These do not establish bond formation, stereochemistry or a transition state.

### Mathematics: pair screening, orientation and shell selection
`REACTIVE_RADIUS` $R$ (default 3.25 Å) screens the **whole CPD COM** against the mass-weighted reactive C=C center:
$$\min_{\mathbf n\in\mathbb Z^3}\|\mathbf R_{\rm CPD}+\mathbf nH-\mathbf R_{\rm CC}\|<R.$$
This is not a forming-bond distance. `MANUAL_SHIFT` / returned `shift` translates CPD by $\mathbf nH$; manual pair selection bypasses automatic screening.

Let $\mathbf a_1,\mathbf a_2$ be CPD terminal carbons and $\mathbf b_1,\mathbf b_2$ the dienophile C=C carbons in the jointly placed geometry. For the two matchings $\pi$ compute $d_i=\|\mathbf a_i-\mathbf b_{\pi(i)}\|$. Choose the smallest lexicographic tuple $(\max(d_1,d_2),d_1+d_2,\text{bond IDs})$. `forming_bonds` records those proposed atom pairs; its local variants change indices, not geometry.

Define $\mathbf u=\mathbf a_2-\mathbf a_1$, $\mathbf v=\mathbf b_2-\mathbf b_1$, $\mathbf s=(\mathbf b_1+\mathbf b_2-\mathbf a_1-\mathbf a_2)/2$. For each four-carbon reactive plane, center its coordinates into $X$ and take the last right singular vector of $X=U\Sigma V^T$ as its unit normal $\mathbf n$. Then
$$A(\mathbf p,\mathbf q)=\frac{180}{\pi}\arccos\!\left(\frac{|\mathbf p\cdot\mathbf q|}{\|\mathbf p\|\|\mathbf q\|}\right),$$
$$\theta_{\rm axis}=A(\mathbf u,\mathbf v),\quad\theta_{\rm plane}=A(\mathbf n_{\rm CPD},\mathbf n_{\rm MAL}),\quad\theta_{\rm approach}=A(\mathbf n_{\rm CPD},\mathbf s).$$
The implementation clips the cosine to [0,1]; these unoriented angles lie in [0°,90°]. Degenerate geometry is rejected. `CRITERIA` / `PAIR_CRITERIA` compare $\max d_i$, $|d_1-d_2|$, and each angle with their corresponding `max_*` fields (Å or degrees). Equality passes; `None` disables that numerical test. They do not establish a transition state or stereochemistry.

`SHELL_RADIUS` $R_s$ retains a **whole neighboring molecule** if an atom in a suitable periodic image lies within $R_s$ of any pair atom:
$$\min_{i\in\mathrm{shell\ molecule},\,j\in\mathrm{pair},\,\mathbf n}\|\mathbf r_i+\mathbf nH-\mathbf r_j\|\le R_s.$$
Pair force rows are copied by persistent ID without rescaling or recomputation. `SELECTED_CANDIDATE` selects a table row, not an atom ID.


In [ ]:
if MANUAL_PAIR_IDS is None:
    pairs, shifts = cpd_pairs.get_all_prereactive_pairs(atoms, scale=BOND_SCALE, reactive_radius=REACTIVE_RADIUS)
    assessments = assess_prereactive_pairs(atoms, pairs, shifts, criteria=CRITERIA, scale=BOND_SCALE)
    display(pd.DataFrame([
        {"candidate":i, "accepted":a.accepted, "reasons":"; ".join(a.reasons),
         **(asdict(a.geometry) if a.geometry is not None else {})}
        for i,a in enumerate(assessments)]))
else:
    pairs, shifts, assessments = [MANUAL_PAIR_IDS], [MANUAL_SHIFT], None
if not 0 <= SELECTED_CANDIDATE < len(pairs):
    raise ValueError("No such candidate: inspect the table or supply a manual pair")

def custom_check(atoms, pair_ids, shift, assessment):
    return True  # replace with your own test; no callback API is assumed

assessment = assessments[SELECTED_CANDIDATE] if assessments is not None else None
if assessment is not None and not assessment.accepted:
    raise ValueError(f"Selected candidate failed: {assessment.reasons}")
if not custom_check(atoms, pairs[SELECTED_CANDIDATE], shifts[SELECTED_CANDIDATE], assessment):
    raise ValueError("Selected candidate failed your custom check")


## Extract whole molecules and check ordering
The shell cutoff is atom-to-pair distance. Every touched molecule is included completely and placed in a consistent periodic image. The result is finite and nonperiodic.

Legacy role labels CPD/MAL denote first/second molecules even in manual mode; they are not chemical identification for other systems.

In [ ]:
clusters = structures.get_pair_shell_clusters(
    atoms, [pairs[SELECTED_CANDIDATE]], [shifts[SELECTED_CANDIDATE]],
    shell_radius=SHELL_RADIUS, scale=BOND_SCALE)
records = structures.get_cluster_positions_and_forces(atoms, frame, clusters)
buffer = StringIO()
structures.pair_data_to_extxyz(records, buffer)
buffer.seek(0)
cluster = read(buffer, format="extxyz")
pair = cluster[cluster.arrays["pair_role"] != "shell"]
lookup = {int(i): row for row,i in enumerate(atoms.arrays["global_index"])}
source_rows = [lookup[int(i)] for i in cluster.arrays["global_index"]]
if "lammps_forces" in cluster.arrays:
    np.testing.assert_allclose(cluster.arrays["lammps_forces"], atoms.arrays["lammps_forces"][source_rows], atol=1e-8)
display(pd.DataFrame({"pair_row":np.arange(len(pair)), "global_index":pair.arrays["global_index"],
                      "role":pair.arrays["pair_role"], "element":pair.get_chemical_symbols()}))
print("Pair / shell atoms:", len(pair), len(cluster)-len(pair))


### Mathematics: alignment and visualization units
Alignment matches persistent IDs and minimizes the equal-weight objective
$$\mathrm{RMSD}=\sqrt{\frac1N\sum_i\|\mathbf r_iQ+\mathbf t-\mathbf r_i^{\rm ref}\|^2}.$$
Centered coordinates give $X^TY=U\Sigma V^T$, $Q=U\,\mathrm{diag}(1,1,\det(UV^T))V^T$, $\mathbf t=\bar{\mathbf r}^{\rm ref}-\bar{\mathbf r}Q$. Coordinates transform as $\mathbf r'=\mathbf rQ+\mathbf t$; vectors as $\mathbf F'=\mathbf FQ$, $\mathbf v'=\mathbf vQ$ (no translation). RMSD is Å. A supplied reference changes orientation, not physical geometry.

Psience coordinates are Bohr: $\mathbf r_{\rm Bohr}=\mathbf r_{\mathrm{Angstrom}}/a_0$, $a_0\approx0.529177$ Å. The ASE-to-Psience conversion performs this step. A force arrow has displacement $\Delta\mathbf r_{\mathrm{Angstrom}}=\lambda\mathbf F$ with `FORCE_SCALE` / `force_scale` $\lambda$ in Å²/eV; arrow endpoints are converted to Bohr too. It is a display scale, not a force modification. `plot_pair_geometry` stores the Psience molecule in Bohr but explicitly plots it back in Å: its `disk_radius`, disk centers and line endpoints are all in that Å figure frame. Disks use fitted plane normals and site midpoints, with a line along $\mathbf s$. `transparency` only affects appearance. No bond-inference behavior is changed by these explanations.


In [ ]:
display(plot_cluster(cluster))
display(plot_cluster(pair, forces="lammps_forces" in pair.arrays))
if assessment is not None and assessment.geometry is not None:
    display(plot_pair_geometry(pair, disk_radius=2., transparency=.5, scale=BOND_SCALE))
if REFERENCE_EXTXYZ is not None:
    aligned, rmsd, transform = align_geometry(pair, read(REFERENCE_EXTXYZ))
    print("Aligned RMSD (angstrom):", rmsd)
    display(plot_cluster(aligned, forces="lammps_forces" in aligned.arrays))


ASE positions are angstrom; conversion to Psience Molecule uses Bohr. Plots and overlays use angstrom. Bonds retain Psience defaults. Alignment rotates geometry and force vectors together; original coordinates remain the export source.

### Mathematics: relative motion and projected forces
For the two molecular COMs, set $\hat{\mathbf e}=(\mathbf R_B-\mathbf R_A)/\|\mathbf R_B-\mathbf R_A\|$:
$$\Delta\mathbf V=\mathbf V_B-\mathbf V_A,\qquad p_A=\mathbf F_A\cdot\hat{\mathbf e},\quad p_B=\mathbf F_B\cdot\hat{\mathbf e},\quad\Delta p=p_B-p_A.$$
COMs and velocities are mass-weighted; molecular forces are sums over atoms. $\Delta\mathbf V$ is Å/ps; $p_A,p_B,\Delta p$ are eV/Å. This difference is not a generalized reaction-coordinate force or an isolated environmental force.


In [ ]:
first_ids, second_ids = pairs[SELECTED_CANDIDATE]
first_center = md.get_molecule_center_of_mass(pair, first_ids, positions_unwrapped=True)
second_center = md.get_molecule_center_of_mass(pair, second_ids, positions_unwrapped=True)
if "momenta" in pair.arrays:
    print("Relative COM velocity (angstrom/ps):",
          md.get_pair_relative_velocity(pair, first_ids, second_ids))
if "lammps_forces" in pair.arrays:
    print("Net-force projections (eV/angstrom):",
          md.get_pair_force_projections(pair, first_ids, second_ids, direction=second_center-first_center))


## Export
Enable WRITE_FILES to create your own inputs for notebook 07. These are total source MD forces, not validated external loads.

In [ ]:
if WRITE_FILES:
    paths = [OUTPUT_DIR/name for name in ("pair.extxyz","cluster.extxyz","selection.json")]
    if not OVERWRITE and any(p.exists() for p in paths):
        raise FileExistsError("Choose a new OUTPUT_DIR or enable OVERWRITE")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write(paths[0], pair)
    write(paths[1], cluster)
    provenance = {"dump":str(files["dump"]), "frame":frame.index, "step":frame.timestep,
                  "units":UNITS, "timestep_ps":TIMESTEP_PS, "shell_radius_angstrom":SHELL_RADIUS,
                  "pair_global_ids":pair.arrays["global_index"].tolist(),
                  "geometry":asdict(assessment.geometry) if assessment and assessment.geometry else None}
    paths[2].write_text(json.dumps(provenance, indent=2, default=lambda x: int(x))+chr(10))
    print("Saved:", OUTPUT_DIR)
